In [2]:
from google.colab import files
uploaded = files.upload()

Saving spam_email_dataset.csv to spam_email_dataset.csv


In [3]:

import pandas as pd
from sklearn.model_selection import train_test_split

# Load uploaded dataset
df = pd.read_csv("spam_email_dataset.csv")

# Split 80% for training and 20% for testing + validation
train_data, temp_data = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

# Split remaining 20% equally
test_data, validate_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    stratify=temp_data["label"]
)

# Save three CSV files
train_data.to_csv("TRAIN.csv", index=False)
test_data.to_csv("TEST.csv", index=False)
validate_data.to_csv("VALIDATE.csv", index=False)

print("DATASET SPLIT COMPLETED!")
print("TRAIN records:", len(train_data))
print("TEST records:", len(test_data))
print("VALIDATE records:", len(validate_data))


DATASET SPLIT COMPLETED!
TRAIN records: 32
TEST records: 4
VALIDATE records: 4


In [4]:
from google.colab import files

files.download("TRAIN.csv")
files.download("TEST.csv")
files.download("VALIDATE.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:

!pip install -q datasets pandas scikit-learn transformers accelerate evaluate

from datasets import load_dataset

# Load the Enron spam email dataset
dataset = load_dataset("SetFit/enron_spam", split="train")

print("Dataset loaded successfully!")
print("Total emails:", len(dataset))
print("Columns:", dataset.column_names)

print("\nFirst email:")
print(dataset[0])


README.md:   0%|          | 0.00/176 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: reconstructing file:   0%|          |  0.00B /  101MB            

train.jsonl: downloading bytes:           |  0.00B            

test.jsonl: reconstructing file:   0%|          |  0.00B / 6.27MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/31716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset loaded successfully!
Total emails: 31716
Columns: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']

First email:
{'message_id': 33214, 'text': 'any software just for 15 $ - 99 $ understanding oem software\nlead me not into temptation ; i can find the way myself .\n# 3533 . the law disregards trifles .', 'label': 1, 'label_text': 'spam', 'subject': 'any software just for 15 $ - 99 $', 'message': 'understanding oem software\nlead me not into temptation ; i can find the way myself .\n# 3533 . the law disregards trifles .', 'date': datetime.datetime(2005, 6, 18, 0, 0)}


In [6]:

import pandas as pd

df = dataset.to_pandas()

print("Label distribution:")
print(df["label_text"].value_counts())

print("\nNumeric labels:")
print(df["label"].value_counts())

print("\nEmail examples:")
display(df[["text", "label", "label_text"]].head(10))


Label distribution:
label_text
spam    16163
ham     15553
Name: count, dtype: int64

Numeric labels:
label
1    16163
0    15553
Name: count, dtype: int64

Email examples:


,text,label,label_text
0,any software just for 15 $ - 99 $ understandin...,1,spam
1,perspective on ferc regulatory action client c...,0,ham
2,wanted to try ci 4 lis but thought it was way ...,1,spam
3,"enron / hpl actuals for december 11 , 2000 tec...",0,ham
4,looking for cheap high - quality software ? ro...,1,spam
5,"emerging growth stock profile vera ,\nvcsc - b...",1,spam
6,internet provider abuse i noticed that you hav...,1,spam
7,[ avfs ] romanian software production & export...,1,spam
8,fortune most admired ranking congratulations !...,0,ham
9,"localized software , all languages available ....",1,spam


In [7]:

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# Load complete dataset
data = load_dataset("SetFit/enron_spam")

train_df = data["train"].to_pandas()
test_df = data["test"].to_pandas()

# Convert labels into binary format
def get_label(value):
    value = str(value).lower().strip()

    if "spam" in value:
        return 1
    elif "ham" in value:
        return 0
    else:
        raise ValueError(f"Unknown label: {value}")

# Prepare columns
train_df["label"] = train_df["label_text"].apply(get_label)
test_df["label"] = test_df["label_text"].apply(get_label)

train_df = train_df[["text", "label"]].dropna()
test_df = test_df[["text", "label"]].dropna()

# Split training data into TRAIN and VALIDATE
train_data, validate_data = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    stratify=train_df["label"]
)

# Save all three files
train_data.to_csv("TRAIN.csv", index=False)
test_df.to_csv("TEST.csv", index=False)
validate_data.to_csv("VALIDATE.csv", index=False)

print("DATASET PREPARATION COMPLETED!")
print("--------------------------------")
print("TRAIN:", len(train_data))
print("TEST:", len(test_df))
print("VALIDATE:", len(validate_data))

print("\nFiles created successfully!")


Repo card metadata block was not found. Setting CardData to empty.


DATASET PREPARATION COMPLETED!
--------------------------------
TRAIN: 28544
TEST: 2000
VALIDATE: 3172

Files created successfully!


In [8]:

from google.colab import files

files.download("TRAIN.csv")
files.download("TEST.csv")
files.download("VALIDATE.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:

!pip install -q transformers accelerate torch scikit-learn pandas


In [3]:

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Select DistilBERT model
model_name = "distilbert/distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pretrained model for spam classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.to(device)

print("DistilBERT loaded successfully!")
print("Model is ready for training.")


Device: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT loaded successfully!
Model is ready for training.


In [5]:

import pandas as pd

base_url = "https://huggingface.co/datasets/Pannuru/spam-email-dataset/resolve/main/"

train_df = pd.read_csv(base_url + "TRAIN.csv")
val_df = pd.read_csv(base_url + "VALIDATE.csv")
test_df = pd.read_csv(base_url + "TEST.csv")

print("All datasets loaded successfully!")

print("TRAIN:", train_df.shape)
print("VALIDATE:", val_df.shape)
print("TEST:", test_df.shape)

print("\nColumns:", train_df.columns.tolist())
display(train_df.head())


All datasets loaded successfully!
TRAIN: (32, 2)
VALIDATE: (4, 2)
TEST: (4, 2)

Columns: ['text', 'label']


,text,label
0,I will call you after I finish the client meet...,0
1,Final notice: pay your outstanding amount usin...,1
2,The department meeting will take place in Room...,0
3,Thanks for your help with the presentation.,0
4,Claim your free vacation by clicking the link ...,1


In [6]:
print("TRAIN:", train_df.shape)
print("VALIDATE:", val_df.shape)
print("TEST:", test_df.shape)

print("\nTraining labels:")
print(train_df["label"].value_counts())

TRAIN: (32, 2)
VALIDATE: (4, 2)
TEST: (4, 2)

Training labels:
label
0    16
1    16
Name: count, dtype: int64


In [7]:

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Prepare dataset
class EmailDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].fillna("").astype(str).tolist()
        self.labels = df["label"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = EmailDataset(train_df)
val_dataset = EmailDataset(val_df)

# Load classification model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Training configuration
training_args = TrainingArguments(
    output_dir="./spam_results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none"
)

# Evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

print("DISTILBERT TRAINING COMPLETED!")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.654327,0.500000,0.500000,1.000000,0.666667
2,No log,0.641372,0.500000,0.500000,1.000000,0.666667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DISTILBERT TRAINING COMPLETED!


In [8]:
import os

print(os.listdir("/content"))

['.config', 'spam_results', 'sample_data']


In [9]:

import os

for root, dirs, files in os.walk("/content/spam_results"):
    print("Folder:", root)
    for file in files:
        print("  File:", file)


Folder: /content/spam_results
Folder: /content/spam_results/checkpoint-4
  File: config.json
  File: scheduler.pt
  File: model.safetensors
  File: trainer_state.json
  File: rng_state.pth
  File: training_args.bin
  File: optimizer.pt
Folder: /content/spam_results/checkpoint-8
  File: config.json
  File: scheduler.pt
  File: model.safetensors
  File: trainer_state.json
  File: rng_state.pth
  File: training_args.bin
  File: optimizer.pt


In [10]:

import json
import os

checkpoint = "/content/spam_results/checkpoint-8"

with open(os.path.join(checkpoint, "trainer_state.json")) as f:
    state = json.load(f)

print("Global training steps:", state["global_step"])
print("Best checkpoint:", state.get("best_model_checkpoint"))
print("Training log:")
for item in state["log_history"][-5:]:
    print(item)



Global training steps: 8
Best checkpoint: None
Training log:
{'epoch': 1.0, 'eval_accuracy': 0.5, 'eval_f1': 0.6666666666666666, 'eval_loss': 0.6543271541595459, 'eval_precision': 0.5, 'eval_recall': 1.0, 'eval_runtime': 0.0414, 'eval_samples_per_second': 96.605, 'eval_steps_per_second': 24.151, 'step': 4}
{'epoch': 2.0, 'eval_accuracy': 0.5, 'eval_f1': 0.6666666666666666, 'eval_loss': 0.6413715481758118, 'eval_precision': 0.5, 'eval_recall': 1.0, 'eval_runtime': 0.0403, 'eval_samples_per_second': 99.356, 'eval_steps_per_second': 24.839, 'step': 8}


In [11]:

import torch
from transformers import TrainingArguments

args = torch.load(
    "/content/spam_results/checkpoint-8/training_args.bin",
    map_location="cpu",
    weights_only=False
)

print("Epochs:", args.num_train_epochs)
print("Batch size:", args.per_device_train_batch_size)
print("Learning rate:", args.learning_rate)
print("Maximum steps:", args.max_steps)
print("Output directory:", args.output_dir)

print("\nFiles in sample_data:")
import os
print(os.listdir("/content/sample_data"))


Epochs: 2
Batch size: 8
Learning rate: 2e-05
Maximum steps: -1
Output directory: ./spam_results

Files in sample_data:
['anscombe.json', 'README.md', 'mnist_test.csv', 'california_housing_test.csv', 'mnist_train_small.csv', 'california_housing_train.csv']


In [12]:

!pip install -q huggingface_hub

from huggingface_hub import HfApi

api = HfApi()
repo_id = "Pannuru/spam-mail-detection"

for repo_type in ["model", "dataset"]:
    print("\nRepository type:", repo_type)
    try:
        files = api.list_repo_files(
            repo_id=repo_id,
            repo_type=repo_type
        )
        print(files)
    except Exception as e:
        print("Not available:", str(e)[:200])



Repository type: model
['.gitattributes', 'README.md', 'SKILL.md', 'analyze.py', 'apply_recovery.py', 'output .json', 'output.jsonl', 'resolve.py']

Repository type: dataset
Not available: 401 Client Error. (Request ID: Root=1-6af58393-55fa40e16e740b24695b7da6;0c3ff10f-d05f-48e4-9139-c2229c8858b2)

Repository Not Found for url: https://huggingface.co/api/datasets/Pannuru/spam-mail-detec


In [13]:

!pip install -q datasets

from datasets import load_dataset

dataset = load_dataset("SetFit/enron_spam")

print(dataset)
print("\nTraining columns:")
print(dataset["train"].column_names)

print("\nFirst training example:")
print(dataset["train"][0])


README.md:   0%|          | 0.00/176 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: reconstructing file:   0%|          |  0.00B /  101MB            

train.jsonl: downloading bytes:           |  0.00B            

test.jsonl: reconstructing file:   0%|          |  0.00B / 6.27MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/31716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 31716
    })
    test: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 2000
    })
})

Training columns:
['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']

First training example:
{'message_id': 33214, 'text': 'any software just for 15 $ - 99 $ understanding oem software\nlead me not into temptation ; i can find the way myself .\n# 3533 . the law disregards trifles .', 'label': 1, 'label_text': 'spam', 'subject': 'any software just for 15 $ - 99 $', 'message': 'understanding oem software\nlead me not into temptation ; i can find the way myself .\n# 3533 . the law disregards trifles .', 'date': datetime.datetime(2005, 6, 18, 0, 0)}


In [14]:

from collections import Counter

print("Training label distribution:")
print(Counter(dataset["train"]["label"]))

print("\nTest label distribution:")
print(Counter(dataset["test"]["label"]))

print("\nLabel examples:")
for i in range(5):
    print("Label:", dataset["train"][i]["label"])
    print("Label text:", dataset["train"][i]["label_text"])
    print("Email:", dataset["train"][i]["text"][:150])
    print("-" * 50)


Training label distribution:
Counter({1: 16163, 0: 15553})

Test label distribution:
Counter({1: 1008, 0: 992})

Label examples:
Label: 1
Label text: spam
Email: any software just for 15 $ - 99 $ understanding oem software
lead me not into temptation ; i can find the way myself .
# 3533 . the law disregards tri
--------------------------------------------------
Label: 0
Label text: ham
Email: perspective on ferc regulatory action client conf call today , jun e 19 th , 2 : 00 pm edt
perspective on ferc regulatory action client conference cal
--------------------------------------------------
Label: 1
Label text: spam
Email: wanted to try ci 4 lis but thought it was way too expensive for you ? viagra at $ 1 . 12 per dose
ready to boost your sex life ? positive ?
time to do
--------------------------------------------------
Label: 0
Label text: ham
Email: enron / hpl actuals for december 11 , 2000 teco tap 30 . 000 / enron ; 120 . 000 / hpl gas daily
ls hpl lsk ic 30 . 000 / enron
-------

In [15]:

!pip install -q transformers accelerate scikit-learn

import torch
import numpy as np
from datasets import DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. Check GPU
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")

# 2. Prepare training data
small_train = dataset["train"].shuffle(seed=42).select(range(8000))
test_data = dataset["test"]

# 3. Load DistilBERT
model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "HAM", 1: "SPAM"},
    label2id={"HAM": 0, "SPAM": 1}
)

# 4. Tokenize emails
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

train_tokenized = small_train.map(
    tokenize_function, batched=True
)

test_tokenized = test_data.map(
    tokenize_function, batched=True
)

train_tokenized = train_tokenized.rename_column("label", "labels")
test_tokenized = test_tokenized.rename_column("label", "labels")

# Remove unnecessary columns
train_tokenized = train_tokenized.remove_columns(
    [c for c in train_tokenized.column_names
     if c not in ["input_ids", "attention_mask", "labels"]]
)

test_tokenized = test_tokenized.remove_columns(
    [c for c in test_tokenized.column_names
     if c not in ["input_ids", "attention_mask", "labels"]]
)

# 5. Evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# 6. Training settings
training_args = TrainingArguments(
    output_dir="/content/truelens_spam_model",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# 7. Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=DataCollatorWithPadding(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# 8. Train
trainer.train()

# 9. Evaluate
results = trainer.evaluate()

print("\nFINAL MODEL RESULTS")
for key, value in results.items():
    print(key, ":", value)

# 10. Save final model
trainer.save_model("/content/final_spam_model")
tokenizer.save_pretrained("/content/final_spam_model")

print("\nMODEL SAVED SUCCESSFULLY!")


Device: GPU


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.120651,0.062631,0.984000,0.973786,0.995040,0.984298
2,0.027027,0.062456,0.986500,0.987090,0.986111,0.986600


... (2 outputs left out)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.027027,0.062456,2,0.986500,0.987090,0.986111,0.986600



FINAL MODEL RESULTS
eval_loss : 0.062456145882606506
eval_accuracy : 0.9865
eval_precision : 0.9870903674280039
eval_recall : 0.9861111111111112
eval_f1 : 0.9866004962779157


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


MODEL SAVED SUCCESSFULLY!


In [16]:
small_train = dataset["train"].shuffle(seed=42).select(range(8000))
test_data = dataset["test"]

print("Training samples:", len(small_train))
print("Testing samples:", len(test_data))

Training samples: 8000
Testing samples: 2000


In [17]:
from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("DistilBERT tokenizer loaded successfully!")

DistilBERT tokenizer loaded successfully!


In [18]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

train_tokenized = small_train.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_data.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed!")
print("Training data:", train_tokenized)
print("Testing data:", test_tokenized)

Tokenization completed!
Training data: Dataset({
    features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8000
})
Testing data: Dataset({
    features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})


In [19]:

import torch
import inspect
import numpy as np

from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

# 2. Prepare tokenized datasets
train_final = train_tokenized.remove_columns(
    [c for c in train_tokenized.column_names
     if c not in ["input_ids", "attention_mask", "label"]]
)

test_final = test_tokenized.remove_columns(
    [c for c in test_tokenized.column_names
     if c not in ["input_ids", "attention_mask", "label"]]
)

train_final = train_final.rename_column("label", "labels")
test_final = test_final.rename_column("label", "labels")

# 3. Load pretrained DistilBERT
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=2,
    id2label={0: "HAM", 1: "SPAM"},
    label2id={"HAM": 0, "SPAM": 1}
)

print("DistilBERT model loaded successfully!")

# 4. Define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# 5. Training configuration
args_dict = {
    "output_dir": "/content/final_spam_model",
    "num_train_epochs": 2,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "learning_rate": 2e-5,
    "save_strategy": "no",
    "report_to": "none",
    "fp16": torch.cuda.is_available()
}

# Support different Transformers versions
if "eval_strategy" in inspect.signature(TrainingArguments).parameters:
    args_dict["eval_strategy"] = "epoch"
else:
    args_dict["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**args_dict)

# 6. Create Trainer
trainer_params = {
    "model": model,
    "args": training_args,
    "train_dataset": train_final,
    "eval_dataset": test_final,
    "data_collator": DataCollatorWithPadding(tokenizer),
    "compute_metrics": compute_metrics
}

if "processing_class" in inspect.signature(Trainer).parameters:
    trainer_params["processing_class"] = tokenizer
else:
    trainer_params["tokenizer"] = tokenizer

trainer = Trainer(**trainer_params)

# 7. Start training
print("\nStarting DistilBERT training...")

trainer.train()

# 8. Evaluate
print("\nMODEL EVALUATION")
results = trainer.evaluate()

for key, value in results.items():
    print(key, ":", value)

# 9. Save model
trainer.save_model("/content/final_spam_model")
tokenizer.save_pretrained("/content/final_spam_model")

print("\nDistilBERT training completed!")
print("Model saved successfully!")


Training device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded successfully!

Starting DistilBERT training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.118090,0.062996,0.984500,0.975657,0.994048,0.984767
2,0.024575,0.068223,0.984500,0.988989,0.980159,0.984554



MODEL EVALUATION


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.024575,0.068223,2,0.984500,0.988989,0.980159,0.984554


eval_loss : 0.06822332739830017
eval_accuracy : 0.9845
eval_precision : 0.988988988988989
eval_recall : 0.9801587301587301
eval_f1 : 0.9845540607872446


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


DistilBERT training completed!
Model saved successfully!


In [20]:
import torch

def predict_email(email):
    inputs = tokenizer(
        email,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)[0]

    prediction = torch.argmax(probabilities).item()

    label = "SPAM" if prediction == 1 else "HAM"

    print("Email:", email)
    print("Prediction:", label)
    print("Ham probability:", round(probabilities[0].item() * 100, 2), "%")
    print("Spam probability:", round(probabilities[1].item() * 100, 2), "%")

# Test examples
predict_email("Congratulations! You won a free prize. Click here to claim now!")
print("\n" + "-" * 50 + "\n")
predict_email("Please find attached the meeting schedule for tomorrow.")

Email: Congratulations! You won a free prize. Click here to claim now!
Prediction: SPAM
Ham probability: 1.7 %
Spam probability: 98.3 %

--------------------------------------------------

Email: Please find attached the meeting schedule for tomorrow.
Prediction: HAM
Ham probability: 99.96 %
Spam probability: 0.04 %


In [21]:

import re

spam_indicators = {
    "Prize and rewards": [
        "won", "winner", "prize", "reward", "free gift"
    ],
    "Urgency": [
        "urgent", "immediately", "act now", "limited time"
    ],
    "Money offers": [
        "cash", "lottery", "claim money", "earn money"
    ],
    "Suspicious links": [
        "click here", "claim now", "verify account"
    ],
    "Promotional language": [
        "buy now", "special offer", "discount", "congratulations"
    ]
}

def explain_email(email):
    inputs = tokenizer(
        email,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0]

    prediction = torch.argmax(probs).item()

    print("PREDICTION:", "SPAM" if prediction == 1 else "HAM")
    print("SPAM PROBABILITY:", round(probs[1].item() * 100, 2), "%")
    print("\nWHY THIS EMAIL MAY BE SPAM:")

    found = []

    email_lower = email.lower()

    for category, words in spam_indicators.items():
        matches = [word for word in words if word in email_lower]

        if matches:
            found.append((category, matches))

    if prediction == 1:
        if found:
            for category, matches in found:
                print("-", category, ":", ", ".join(matches))
        else:
            print("- The model detected spam-like language,")
            print("  but no predefined warning phrases were found.")
    else:
        print("The model classified this email as legitimate.")
        if found:
            print("Note: Some promotional phrases were found,")
            print("but their presence alone does not prove spam.")

    print("\nNote: These are keyword-based clues,")
    print("not a causal explanation of the model's decision.")

# Test
email = "Congratulations! You won a free prize. Click here to claim now!"

explain_email(email)


PREDICTION: SPAM
SPAM PROBABILITY: 98.3 %

WHY THIS EMAIL MAY BE SPAM:
- Prize and rewards : won, prize
- Suspicious links : click here, claim now
- Promotional language : congratulations

Note: These are keyword-based clues,
not a causal explanation of the model's decision.


In [22]:
import os

SAVE_PATH = "/content/final_spam_model"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")
print("Files:", os.listdir(SAVE_PATH))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
Files: ['config.json', 'tokenizer_config.json', 'model.safetensors', 'tokenizer.json', 'training_args.bin']


In [23]:
from huggingface_hub import notebook_login

notebook_login()

In [24]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/final_spam_model",
    repo_id="Pannuru/spam-mail-detection",
    repo_type="model"
)

print("Model uploaded successfully!")

Model uploaded successfully!


In [25]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

repo_id = "Pannuru/spam-mail-detection"

tokenizer_hf = AutoTokenizer.from_pretrained(repo_id)
model_hf = AutoModelForSequenceClassification.from_pretrained(repo_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_hf.to(device)
model_hf.eval()

def test_email(email):
    inputs = tokenizer_hf(
        email,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model_hf(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0]

    prediction = torch.argmax(probs).item()

    print("Email:", email)
    print("Result:", "SPAM" if prediction == 1 else "HAM")
    print("Spam probability:", round(probs[1].item()*100, 2), "%")

test_email("Congratulations! You won a free gift. Claim now!")

config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Email: Congratulations! You won a free gift. Claim now!
Result: SPAM
Spam probability: 99.82 %


In [26]:

from huggingface_hub import HfApi

readme = """
# Spam Email Detection Using DistilBERT

## Project Overview
This project uses DistilBERT, a transformer-based language
model, to classify emails as Spam or Ham.

## Objectives
- Detect spam and legitimate emails.
- Classify email messages automatically.
- Identify suspicious words and phrases.
- Provide explanations for potential spam indicators.

## Dataset
Dataset: SetFit/enron_spam

- Training samples: 8,000
- Testing samples: 2,000
- Spam label: 1
- Ham label: 0

## Technology Used
- Python
- Google Colab
- Hugging Face Transformers
- DistilBERT
- PyTorch
- Scikit-learn

## Methodology
1. Dataset loading
2. Data preparation
3. Tokenization
4. DistilBERT fine-tuning
5. Model evaluation
6. Spam and Ham prediction
7. Spam indicator explanation

## Model
Architecture: DistilBERT
Maximum sequence length: 128
Training epochs: 2

## Example
Input: Congratulations! You won a free prize.

Prediction: SPAM

## Limitations
The explanation feature identifies predefined suspicious
phrases. These phrases are indicators, not proof of spam
or a complete explanation of the model's decision.

## Model Repository
https://huggingface.co/Pannuru/spam-mail-detection
"""

api = HfApi()

api.upload_file(
    path_or_fileobj=readme.encode("utf-8"),
    path_in_repo="README.md",
    repo_id="Pannuru/spam-mail-detection",
    repo_type="model",
    commit_message="Add project documentation"
)

print("README uploaded successfully!")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:11779: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


README uploaded successfully!


In [27]:
from huggingface_hub import HfApi

readme = """---
library_name: transformers
pipeline_tag: text-classification
tags:
- distilbert
- spam-detection
- email-classification
- text-classification
---

# Spam Email Detection Using DistilBERT

## Project Overview
A DistilBERT-based model that classifies email messages as Spam or Ham.

## Dataset
SetFit/enron_spam

- Training samples: 8,000
- Testing samples: 2,000
- Spam label: 1
- Ham label: 0

## Technologies
- Python
- PyTorch
- Hugging Face Transformers
- Google Colab
- DistilBERT

## Workflow
1. Dataset preparation
2. Tokenization
3. Model training
4. Model evaluation
5. Spam and Ham prediction
6. Spam indicator explanation

## Example
Input: Congratulations! You won a free prize.

Prediction: SPAM

## Limitation
Spam explanations use predefined keyword indicators and do not establish the model's causal reasoning.
"""

api = HfApi()

api.upload_file(
    path_or_fileobj=readme.encode("utf-8"),
    path_in_repo="README.md",
    repo_id="Pannuru/spam-mail-detection",
    repo_type="model",
    commit_message="Add Hugging Face model metadata"
)

print("README metadata updated!")

README metadata updated!


In [28]:

import pandas as pd

# Create separate datasets
train_data = small_train
validate_data = dataset["train"].shuffle(seed=42).select(range(8000, 10000))
test_data = dataset["test"]

# Convert to DataFrames
train_df = train_data.to_pandas()
validate_df = validate_data.to_pandas()
test_df = test_data.to_pandas()

# Keep only the required columns
train_df = train_df[["text", "label"]]
validate_df = validate_df[["text", "label"]]
test_df = test_df[["text", "label"]]

# Save CSV files
train_df.to_csv("TRAIN.csv", index=False)
test_df.to_csv("TEST.csv", index=False)
validate_df.to_csv("VALIDATE.csv", index=False)

print("All three files created successfully!")
print("TRAIN:", len(train_df), "emails")
print("TEST:", len(test_df), "emails")
print("VALIDATE:", len(validate_df), "emails")


All three files created successfully!
TRAIN: 8000 emails
TEST: 2000 emails
VALIDATE: 2000 emails


In [29]:

from huggingface_hub import notebook_login, HfApi

# Login to Hugging Face
notebook_login()

# Create dataset repository
api = HfApi()

api.create_repo(
    repo_id="Pannuru/spam-email-dataset",
    repo_type="dataset",
    exist_ok=True
)

# Upload the three CSV files
for file_name in ["TRAIN.csv", "TEST.csv", "VALIDATE.csv"]:
    api.upload_file(
        path_or_fileobj=file_name,
        path_in_repo=file_name,
        repo_id="Pannuru/spam-email-dataset",
        repo_type="dataset"
    )
    print(file_name, "uploaded successfully!")

print("All dataset files uploaded!")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/TRAIN.csv          :   7%|7         |  844kB / 11.6MB            

Validating                    :           |   0%            

TRAIN.csv uploaded successfully!
TEST.csv uploaded successfully!
VALIDATE.csv uploaded successfully!
All dataset files uploaded!


In [31]:

from huggingface_hub import HfApi

api = HfApi()

files = api.list_repo_files(
    repo_id="Pannuru/spam-email-dataset",
    repo_type="dataset"
)

print("Files currently in your repository:")
for file in files:
    print(file)


Files currently in your repository:
.gitattributes
README.md
TEST (1).csv
TEST.csv
TRAIN (1).csv
TRAIN.csv
VALIDATE (1).csv
VALIDATE.csv


In [32]:

from huggingface_hub import HfApi

api = HfApi()

# List all files in your dataset repository
files = api.list_repo_files(
    repo_id="Pannuru/spam-email-dataset",
    repo_type="dataset"
)

print("FILES IN HUGGING FACE REPOSITORY:")
for file in files:
    print(" -", file)


FILES IN HUGGING FACE REPOSITORY:
 - .gitattributes
 - README.md
 - TEST (1).csv
 - TEST.csv
 - TRAIN (1).csv
 - TRAIN.csv
 - VALIDATE (1).csv
 - VALIDATE.csv


In [33]:

from huggingface_hub import hf_hub_download
import pandas as pd

repo_id = "Pannuru/spam-email-dataset"

files = [
    "TRAIN.csv", "TRAIN (1).csv",
    "TEST.csv", "TEST (1).csv",
    "VALIDATE.csv", "VALIDATE (1).csv"
]

for file in files:
    path = hf_hub_download(
        repo_id=repo_id,
        filename=file,
        repo_type="dataset"
    )

    df = pd.read_csv(path)

    print(file, ":", len(df), "records")


TRAIN.csv: reconstructing file:   0%|          |  0.00B / 11.6MB            

TRAIN.csv: downloading bytes:           |  0.00B            

TRAIN.csv : 8000 records


TRAIN (1).csv: reconstructing file:   0%|          |  0.00B / 42.9MB            

TRAIN (1).csv: downloading bytes:           |  0.00B            

TRAIN (1).csv : 28544 records


... (2 outputs left out)

VALIDATE.csv:   0%|          | 0.00/2.85M [00:00<?, ?B/s]

VALIDATE.csv : 2000 records


VALIDATE%20%281%29.csv:   0%|          | 0.00/5.14M [00:00<?, ?B/s]

VALIDATE (1).csv : 3172 records


In [34]:

from huggingface_hub import HfApi

api = HfApi()

duplicate_files = [
    "TRAIN (1).csv",
    "TEST (1).csv",
    "VALIDATE (1).csv"
]

for file in duplicate_files:
    api.delete_file(
        path_in_repo=file,
        repo_id="Pannuru/spam-email-dataset",
        repo_type="dataset",
        commit_message=f"Remove extra file {file}"
    )
    print(file, "deleted successfully!")

print("Duplicate files removed!")


TRAIN (1).csv deleted successfully!
TEST (1).csv deleted successfully!
VALIDATE (1).csv deleted successfully!
Duplicate files removed!
